In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

In [3]:
drive_path = Path("/content/drive/MyDrive/poc/artifacts")

In [4]:
datasets = [d for d in os.listdir(drive_path) if os.path.isdir(drive_path / d)]

models = [
    "BiasedSVD",
    "GCR",
    "FuzzGCR-product",
    "FuzzGCR-godel",
    "FuzzGCR-smooth",
    "FeatureAwareGCR",
]

In [5]:
def group_experiments(experiments: list[str]) -> dict[str, list[str]]:
    grouped_experiments = {
        model: [exp for exp in experiments if exp.startswith(model)] for model in models
    }
    feature_aware_experiments = grouped_experiments.pop("FeatureAwareGCR")
    grouped_experiments["FeatureAwareGCRWithIdEmbed"] = [
        exp for exp in feature_aware_experiments if "embed_id-True" in exp
    ]
    grouped_experiments["FeatureAwareGCRWithoutIdEmbed"] = [
        exp for exp in feature_aware_experiments if "embed_id-False" in exp
    ]
    return grouped_experiments

def find_best_model(dataset: str, model: str, experiments: list[str]) -> str:
    records = []
    for exp in tqdm(experiments, desc=f"Dataset: {dataset} - Model: {model}"):
        record = record = {"model": model, "experiment": exp}
        metrics_path = drive_path / dataset / exp / "metrics.json"
        if not metrics_path.exists():
            continue
        with open(metrics_path, "r") as f:
            metrics = json.load(f)
        record.update(metrics["hparams"])
        record.update(metrics["metrics"])
        records.append(record)
    best_experiment = pd.DataFrame.from_records(records).sort_values(
        by="NDCG@5", ascending=False
    ).iloc[0]
    return best_experiment["experiment"]

def find_best_models(dataset: str) -> dict[str, str]:
    dataset_path = drive_path / dataset
    experiments = os.listdir(dataset_path)
    grouped_experiments = group_experiments(experiments)
    best_models = {
        model: find_best_model(dataset, model, experiments)
        for model, experiments in grouped_experiments.items()
        if len(experiments) > 0
    }
    return best_models

best_models = {
    dataset: find_best_models(dataset) for dataset in datasets
}


Dataset: ml-1m - Model: FuzzGCR-smooth: 100%|██████████| 18/18 [00:00<00:00, 207.76it/s]
Dataset: ml-1m - Model: FeatureAwareGCRWithIdEmbed: 100%|██████████| 20/20 [00:00<00:00, 257.84it/s]
Dataset: ml-1m - Model: FeatureAwareGCRWithoutIdEmbed: 100%|██████████| 17/17 [00:00<00:00, 204.32it/s]
Dataset: ml-100k - Model: FuzzGCR-smooth: 100%|██████████| 18/18 [00:00<00:00, 246.54it/s]
Dataset: ml-100k - Model: FeatureAwareGCRWithIdEmbed: 100%|██████████| 20/20 [00:00<00:00, 200.38it/s]
Dataset: ml-100k - Model: FeatureAwareGCRWithoutIdEmbed: 100%|██████████| 20/20 [00:00<00:00, 255.52it/s]


In [6]:
best_models_path = drive_path / "best_models.json"
with open(best_models_path, "w") as f:
    json.dump(best_models, f, indent=4)

In [7]:
def load_predictions(dataset: str, model: str) -> pd.DataFrame:
    predictions_path = drive_path / dataset / model / "test_predictions.csv"
    return pd.read_csv(predictions_path)

def compute_ranks(pos_scores: np.array, neg_scores: np.array) -> np.array:
    pos_scores = pos_scores.reshape(-1, 1)
    ranks = (neg_scores >= pos_scores).sum(axis=1) + 1
    return ranks

def hit_at_k(ranks: np.array, k: int) -> np.array:
    return (ranks <= k).astype(int)

def ndcg_at_k(ranks: np.array, k: int) -> np.array:
    dcg = 1.0 / np.log2(ranks + 1)
    return np.where(ranks <= k, dcg, 0.0)

def save_metrics(
    dataset: str,
    model: str,
    hits_at_1: np.array,
    hits_at_5: np.array,
    hits_at_10: np.array,
    ndcgs_at_5: np.array,
    ndcgs_at_10: np.array
) -> None:
    metrics_path = drive_path / dataset / f"Best{model}Metrics.json"
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    metrics = {
        "NDCG@5": ndcgs_at_5.tolist(),
        "NDCG@10": ndcgs_at_10.tolist(),
        "HIT@1": hits_at_1.tolist(),
        "HIT@5": hits_at_5.tolist(),
        "HIT@10": hits_at_10.tolist(),
    }
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)

def compute_metrics(dataset: str, model: str, best_model: str) -> dict[str, float]:
    predictions = load_predictions(dataset, best_model)
    neg_cols = [col for col in predictions.columns if col.startswith("neg_pred_")]
    ranks = compute_ranks(predictions["pos_pred"].values, predictions[neg_cols].values)
    hits_at_1 = hit_at_k(ranks, 1)
    hits_at_5 = hit_at_k(ranks, 5)
    hits_at_10 = hit_at_k(ranks, 10)
    ndcgs_at_5 = ndcg_at_k(ranks, 5)
    ndcgs_at_10 = ndcg_at_k(ranks, 10)
    save_metrics(dataset, model, hits_at_1, hits_at_5, hits_at_10, ndcgs_at_5, ndcgs_at_10)
    return {
        "Dataset": dataset,
        "Model": model,
        "HIT@1": hits_at_1.mean(),
        "HIT@5": hits_at_5.mean(),
        "HIT@10": hits_at_10.mean(),
        "NDCG@5": ndcgs_at_5.mean(),
        "NDCG@10": ndcgs_at_10.mean(),
    }

metrics = []
for dataset, models in best_models.items():
    for model, best_model in models.items():
        metrics.append(compute_metrics(dataset, model, best_model))
metrics_df = pd.DataFrame.from_records(metrics)

In [8]:
metrics_df

,Dataset,Model,HIT@1,HIT@5,HIT@10,NDCG@5,NDCG@10
0,beauty,BiasedSVD,0.076868,0.214327,0.312615,0.146682,0.178390
1,beauty,GCR,0.252649,0.355095,0.438447,0.303940,0.330609
2,beauty,FuzzGCR-product,0.021509,0.074409,0.132048,0.047683,0.066131
3,beauty,FuzzGCR-godel,0.056477,0.172830,0.261772,0.114709,0.143335
4,beauty,FuzzGCR-smooth,0.059116,0.170326,0.255690,0.115121,0.142539
5,clothing,BiasedSVD,0.054967,0.166882,0.254170,0.111376,0.139414
6,clothing,GCR,0.306548,0.350750,0.376038,0.329512,0.337637
7,clothing,FuzzGCR-product,0.014091,0.070328,0.136543,0.041458,0.062575
8,clothing,FuzzGCR-godel,0.053114,0.158606,0.243710,0.106542,0.133893
9,clothing,FuzzGCR-smooth,0.051464,0.158555,0.240714,0.105467,0.131854


In [9]:
metrics_df.to_latex(index=False, float_format="%.4f")

'\\begin{tabular}{llrrrrr}\n\\toprule\nDataset & Model & HIT@1 & HIT@5 & HIT@10 & NDCG@5 & NDCG@10 \\\\\n\\midrule\nbeauty & BiasedSVD & 0.0769 & 0.2143 & 0.3126 & 0.1467 & 0.1784 \\\\\nbeauty & GCR & 0.2526 & 0.3551 & 0.4384 & 0.3039 & 0.3306 \\\\\nbeauty & FuzzGCR-product & 0.0215 & 0.0744 & 0.1320 & 0.0477 & 0.0661 \\\\\nbeauty & FuzzGCR-godel & 0.0565 & 0.1728 & 0.2618 & 0.1147 & 0.1433 \\\\\nbeauty & FuzzGCR-smooth & 0.0591 & 0.1703 & 0.2557 & 0.1151 & 0.1425 \\\\\nclothing & BiasedSVD & 0.0550 & 0.1669 & 0.2542 & 0.1114 & 0.1394 \\\\\nclothing & GCR & 0.3065 & 0.3508 & 0.3760 & 0.3295 & 0.3376 \\\\\nclothing & FuzzGCR-product & 0.0141 & 0.0703 & 0.1365 & 0.0415 & 0.0626 \\\\\nclothing & FuzzGCR-godel & 0.0531 & 0.1586 & 0.2437 & 0.1065 & 0.1339 \\\\\nclothing & FuzzGCR-smooth & 0.0515 & 0.1586 & 0.2407 & 0.1055 & 0.1319 \\\\\nml-1m & BiasedSVD & 0.1008 & 0.3232 & 0.4699 & 0.2135 & 0.2607 \\\\\nml-1m & GCR & 0.4482 & 0.4623 & 0.4709 & 0.4552 & 0.4580 \\\\\nml-1m & FuzzGCR-product 